<a href="https://colab.research.google.com/github/LegalIntermediaSL/Nautica/blob/main/simulaciones/14_lectura_grib.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Simulación 14: Lectura Interpretativa de un Campo de Viento (estilo GRIB/Windy)

Los archivos **GRIB** (GRIdded Binary) son el formato estándar mundial en el que los servicios meteorológicos (ECMWF, NOAA/GFS, AEMET...) distribuyen las salidas de sus modelos numéricos de predicción. Aplicaciones como *Windy*, *PredictWind* o *Zygrib* simplemente descargan estos archivos y los "pintan" sobre un mapa: flechas de viento, isobaras (líneas de igual presión) y capas de color por intensidad.

**Importante:** esta simulación **no lee archivos GRIB reales** (para ello harían falta librerías como `pygrib`, `cfgrib` o `xarray` y un archivo binario descargado de un modelo). Es una aproximación didáctica: generamos con `numpy` una rejilla sintética de presión y viento alrededor de un centro de baja presión (borrasca) para entender la **lógica de interpretación visual** que usarás sobre cualquier carta de viento real.

## Fundamento físico simplificado
*   El viento tiende a soplar de forma aproximadamente **paralela a las isobaras** (viento geostrófico), no directamente de la alta a la baja presión.
*   En el **Hemisferio Norte**, el viento circula en sentido **antihorario** alrededor de una borrasca (ciclónico) y horario alrededor de un anticiclón.
*   Cerca de la superficie, el rozamiento hace que el viento se desvíe ligeramente hacia el centro de bajas presiones (converge), en vez de ser puramente tangencial.
*   Cuanto más **juntas** están las isobaras, mayor es el gradiente de presión y **más fuerte sopla el viento** (más flechas/colores intensos).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- REJILLA (equivalente a los puntos de una malla GRIB) ---
nx, ny = 25, 25
x = np.linspace(-500, 500, nx)  # km, eje Este-Oeste
y = np.linspace(-500, 500, ny)  # km, eje Norte-Sur
X, Y = np.meshgrid(x, y)

# --- CENTRO DE LA BORRASCA (Baja presión) ---
cx, cy = 0.0, 0.0
R = np.sqrt((X - cx) ** 2 + (Y - cy) ** 2)
R = np.where(R == 0, 1e-6, R)  # evita división por cero justo en el centro

# --- CAMPO DE PRESIÓN ---
# Modelo radial simplificado: mínimo en el centro, sube hacia el exterior.
P0 = 990       # hPa en el centro de la borrasca
P_amb = 1015   # hPa en el entorno (aire no perturbado)
escala = 300   # km, "radio característico" de la borrasca
Presion = P_amb - (P_amb - P0) * np.exp(-R / escala)

# --- CAMPO DE VIENTO (aprox. geostrófico + convergencia superficial) ---
# Dirección tangencial a las isobaras (antihoraria, Hemisferio Norte)
angulo_tangencial = np.arctan2(Y - cy, X - cx) + np.pi / 2
angulo_convergencia = np.radians(20)  # leve giro hacia el centro por rozamiento
angulo_viento = angulo_tangencial - angulo_convergencia

# Intensidad: crece con el gradiente, máxima en un anillo intermedio
Vmax = 45  # nudos, viento máximo sostenido de la borrasca
Velocidad = Vmax * (R / escala) * np.exp(1 - R / escala)
Velocidad = np.clip(Velocidad, 0, Vmax)

U = Velocidad * np.cos(angulo_viento)  # componente Este-Oeste
V = Velocidad * np.sin(angulo_viento)  # componente Norte-Sur

print("--- CAMPO SINTÉTICO GENERADO ---")
print(f"Presión mínima (centro de la borrasca): {Presion.min():.1f} hPa")
print(f"Presión en el borde de la rejilla:       {Presion.max():.1f} hPa")
print(f"Viento máximo simulado:                  {Velocidad.max():.1f} nudos")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))

# Isobaras: curvas de igual presión, como en cualquier carta sinóptica
niveles = np.arange(990, 1016, 2)
cs = ax.contour(X, Y, Presion, levels=niveles, colors="gray", linewidths=0.8)
ax.clabel(cs, inline=True, fontsize=7, fmt="%d hPa")

# Líneas de corriente coloreadas por intensidad (estilo capa de viento de Windy)
strm = ax.streamplot(X, Y, U, V, color=Velocidad, cmap="turbo", density=1.5, linewidth=1)
fig.colorbar(strm.lines, ax=ax, label="Velocidad del viento (nudos)")

# Flechas (quiver) submuestreadas para no saturar el gráfico
paso = 2
ax.quiver(X[::paso, ::paso], Y[::paso, ::paso], U[::paso, ::paso], V[::paso, ::paso],
          color="black", alpha=0.35, scale=800, width=0.002)

ax.plot(cx, cy, "rx", markersize=14, markeredgewidth=3, label="Centro de baja presión (L)")
ax.set_xlabel("Distancia Este-Oeste (km)")
ax.set_ylabel("Distancia Norte-Sur (km)")
ax.set_title("Campo de Viento Sintético alrededor de una Borrasca (estilo GRIB/Windy)")
ax.legend(loc="upper right")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

## Conclusión

Al interpretar un archivo GRIB real en Windy, PredictWind o el visor de AEMET, la lógica es idéntica a la de este gráfico sintético:
*   **Isobaras muy juntas → viento fuerte.** Isobaras separadas → viento flojo.
*   El viento **no cruza perpendicularmente** las isobaras hacia el centro, sino que gira siguiéndolas (con una ligera componente de convergencia cerca de la superficie).
*   En el Hemisferio Norte una borrasca gira **en sentido antihorario**; un anticiclón, en sentido horario (en el Hemisferio Sur es al revés).

Para trabajar con archivos GRIB reales en Python necesitarías librerías especializadas (`pygrib`, `cfgrib` + `xarray`) y descargar los ficheros del modelo meteorológico (GFS, ECMWF...); esta simulación se limita a reproducir **la lógica visual** de lectura, que es lo que realmente hay que dominar para planificar una travesía.